**Name:** \_\_HAN Zhuo\_\_

**EID:** \_\_zhuohan3\_\_

# CS5489 - Tutorial 5
## Face Detection with CNNs

In the previous tutorial, you used an MLP to detect a face in a small image patch.
In this tutorial you will train a CNN instead of an MLP.

First we need to initialize Python.  Run the below cell.

In [ ]:
%matplotlib inline
import matplotlib_inline   # setup output image format
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')
import matplotlib.pyplot as plt
import matplotlib
from numpy import *
from sklearn import *
import os
import zipfile
import fnmatch
random.seed(100)
import skimage.io
import skimage.color
import skimage.transform
from scipy import ndimage

Next we will load torch.

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
from torchvision import transforms
import sys
print("Python:", sys.version, "PyTorch:", torch.__version__)

## 1. Loading Data and Pre-processing
Next we need to load the images.  Download `faces.zip`, and place it in the same directory as this ipynb file.  **Do not unzip it.** Then run the following cell to load the images.

In [ ]:
imgdata = {'train':[], 'test':[]}
classes = {'train':[], 'test':[]}

# the dataset is too big, so subsample the training and test sets...
# reduce training set by a factor of 4
train_subsample = 1
train_counter = [0, 0]
# maximum number of samples in each class for test set
test_maxsample = 472
test_counter = [0, 0]

# load the zip file
filename = 'faces.zip'
zfile = zipfile.ZipFile(filename, 'r')

for name in zfile.namelist():
    # check file name matches
    if fnmatch.fnmatch(name, "faces/*/*/*.png"):

        # filename is : faces/train/face/fname.png
        (fdir1, fname)  = os.path.split(name)     # get file name
        (fdir2, fclass) = os.path.split(fdir1) # get class (face, nonface)
        (fdir3, fset)   = os.path.split(fdir2) # get training/test set
        # class 1 = face; class 0 = non-face
        myclass = int(fclass == "face")

        loadme = False
        if fset == 'train':
            if (train_counter[myclass] % train_subsample) == 0:
                loadme = True
            train_counter[myclass] += 1
        elif fset == 'test':
            if test_counter[myclass] < test_maxsample:
                loadme = True
            test_counter[myclass] += 1

        if (loadme):
            # open file in memory, and parse as an image
            myfile = zfile.open(name)
            #img = matplotlib.image.imread(myfile)
            img = skimage.io.imread(myfile, as_gray=True)
            myfile.close()

            # append data
            imgdata[fset].append(img)
            classes[fset].append(myclass)


zfile.close()
imgsize = img.shape

print(len(imgdata['train']))
print(len(imgdata['test']))
trainclass2start = sum(classes['train'])

Next we will convert the list of images into a tensor of images for easier processing.

In [ ]:
# convert list to numpy array
trainY = asarray(classes['train'])
testY  = asarray(classes['test'])

# convert class labels to binary indicators
trainYb_np = zeros((len(trainY), 2))
trainYb_np[arange(len(trainY)), trainY] = 1
testYb_np = zeros((len(testY), 2))
testYb_np[arange(len(testY)), testY] = 1
trainYb = F.one_hot(torch.tensor(trainY, dtype=torch.long), num_classes=2).float()
testYb = F.one_hot(torch.tensor(testY, dtype=torch.long), num_classes=2).float()

# convert list of ndarray to ndarray (numpy version)
trainI_np = asarray(imgdata['train']).reshape((6977,19,19,1))
testI_np = asarray(imgdata['test']).reshape((944,19,19,1))
trainI = torch.tensor(trainI_np, dtype=torch.float32).permute(0, 3, 1, 2)  # Convert NHWC to NCHW
testI = torch.tensor(testI_np, dtype=torch.float32).permute(0, 3, 1, 2)   # Convert NHWC to NCHW

# cleanup memory
del imgdata

# shuffle the data (since it is in order by class)
random.seed(123)
inds1 = random.permutation(len(trainI_np)).tolist()
inds2 = random.permutation(len(testI_np)).tolist()
trainYb = trainYb[inds1]
testYb = testYb[inds2]
trainY = trainY[inds1]
testY = testY[inds2]
trainI = trainI[inds1]
testI = testI[inds2]

print(trainI.shape)
print(testI.shape)

Each image is a 19x19x1 array of pixel values.  The last dimension is the number of channels in the image - in this case the image is grayscale, so there is only 1 channel.  Run the below code to show an example:

In [ ]:
print(img.shape)
plt.subplot(1,2,1)
plt.imshow(squeeze(trainI[1]), cmap='gray', interpolation='nearest')
plt.title("face sample")
plt.subplot(1,2,2)
plt.imshow(squeeze(trainI[2]), cmap='gray', interpolation='nearest')
plt.title("non-face sample")
plt.show()

Run the below code to show more images!

In [ ]:
# function to make an image montage
def image_montage(X, imsize=None, maxw=10):
    """X can be a list of images, or a matrix of vectorized images.
      Specify imsize when X is a matrix."""
    tmp = []
    numimgs = len(X)

    # create a list of images (reshape if necessary)
    for i in range(0,numimgs):
        if imsize != None:
            tmp.append(X[i].reshape(imsize))
        else:
            tmp.append(squeeze(X[i]))

    # add blanks
    if (numimgs > maxw) and (mod(numimgs, maxw) > 0):
        leftover = maxw - mod(numimgs, maxw)
        meanimg = 0.5*(X[0].max()+X[0].min())
        for i in range(0,leftover):
            tmp.append(ones(tmp[0].shape)*meanimg)

    # make the montage
    tmp2 = []
    for i in range(0,len(tmp),maxw):
        tmp2.append( hstack(tmp[i:i+maxw]) )
    montimg = vstack(tmp2)
    return montimg

# show images in a plot
def show_imgs(W_list, nc=10, highlight_green=None, highlight_red=None, titles=None):
    # nc is the number of columns
    nfilter = len(W_list)
    nr = (nfilter - 1) // nc + 1
    for i in range(nr):
        for j in range(nc):
            idx = i * nc + j
            if idx == nfilter:
                break
            plt.subplot(nr, nc, idx + 1)
            cur_W = W_list[idx]
            plt.imshow(cur_W,cmap='gray', interpolation='nearest')
            if titles is not None:
                if isinstance(titles, str):
                    plt.title(titles.format(idx))
                else:
                    plt.title(titles[idx])

            if ((highlight_green is not None) and highlight_green[idx]) or \
               ((highlight_red is not None) and highlight_red[idx]):
                ax = plt.gca()
                if highlight_green[idx]:
                    mycol = '#00FF00'
                else:
                    mycol = 'r'
                for S in ['bottom', 'top', 'right', 'left']:
                    ax.spines[S].set_color(mycol)
                    ax.spines[S].set_lw(2.0)
                ax.xaxis.set_ticks_position('none')
                ax.yaxis.set_ticks_position('none')
                ax.set_xticks([])
                ax.set_yticks([])
            else:
                plt.gca().set_axis_off()

# show a few images
plt.figure(figsize=(9,4))
plt.imshow(image_montage(trainI[trainYb[:,0]==1][0:50]), cmap='gray', interpolation='nearest')
plt.show()

plt.figure(figsize=(9,4))
plt.imshow(image_montage(trainI[trainYb[:,1]==1][0:50]), cmap='gray', interpolation='nearest')
plt.show()

Next we will generate the training/validation set from the training data.

In [ ]:
# generate fixed validation set of 10% of the training set
vtrainI, validI, vtrainYb, validYb = \
  model_selection.train_test_split(trainI, trainYb,
  train_size=0.9, test_size=0.1, random_state=4488)

# make validation data
validsetI = (validI, validYb)

print(vtrainI.shape)
print(validI.shape)

Here are some useful functions.

In [ ]:
def plot_history(history):
    fig, ax1 = plt.subplots()

    ax1.plot(history.history['loss'], 'r', label="training loss ({:.6f})".format(history.history['loss'][-1]))
    ax1.plot(history.history['val_loss'], 'r--', label="validation loss ({:.6f})".format(history.history['val_loss'][-1]))
    ax1.grid(True)
    ax1.set_xlabel('iteration')
    ax1.legend(loc="best", fontsize=9)
    ax1.set_ylabel('loss', color='r')
    ax1.tick_params('y', colors='r')

    if 'accuracy' in history.history:
        ax2 = ax1.twinx()

        ax2.plot(history.history['accuracy'], 'b', label="training acc ({:.4f})".format(history.history['accuracy'][-1]))
        ax2.plot(history.history['val_accuracy'], 'b--', label="validation acc ({:.4f})".format(history.history['val_accuracy'][-1]))

        ax2.legend(loc="best", fontsize=9)
        ax2.set_ylabel('acc', color='b')
        ax2.tick_params('y', colors='b')

EarlyStopping function or just use the lightning lib

In [ ]:
# early stopping criteria
class EarlyStopping:
    def __init__(self, monitor='val_accuracy', min_delta=0.0001, patience=5, verbose=1, mode='auto'):
        self.monitor = monitor                         # use validation accuracy for stopping
        self.min_delta = min_delta
        self.patience = patience
        self.verbose = verbose
        self.mode = mode
        self.best_score = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                if self.verbose:
                    print(f"early stopping")
                self.early_stop = True
        else:
            self.best_score = score
            self.counter = 0

 Convert history to match original format

In [ ]:
# Convert history to match original format
class HistoryWrapper:
    def __init__(self, history_dict):
        self.history = history_dict

Basic setting for some trainning parametes

In [ ]:
batch_size = 50
epochs = 100

# Create data loaders
train_dataset = TensorDataset(vtrainI, vtrainYb)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_dataset = TensorDataset(validI, validYb)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)


Now let's try a simple logistic regression classifier, trained using Pytorch. Since the inputs are images, we will convert the input image into a vector using the "Flatten" layer first.

In [ ]:
# initialize random seed
torch.manual_seed(4487)
random.seed(4487)

# build the network for logistic regression
nn_model = nn.Sequential(
    nn.Flatten(),                                     # vectorize the input image
    nn.Linear(19*19*1, 2),                           # classification layer (2 classes)
    nn.Softmax(dim=1)
)

# early stopping criteria
earlystop = EarlyStopping(monitor='val_accuracy', min_delta=0.0001, patience=5, verbose=1, mode='auto')
callbacks_list = [earlystop]

# compile and fit the network
criterion = nn.CrossEntropyLoss()                     # categorical_crossentropy equivalent
optimizer = optim.SGD(nn_model.parameters(), lr=0.05, momentum=0.9, nesterov=True)
                                                      # also calculate accuracy during training

history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}

for epoch in range(epochs):
    # Training phase
    nn_model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = nn_model(batch_x)
        loss = criterion(outputs, torch.argmax(batch_y, dim=1))
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_total += batch_y.size(0)
        train_correct += (predicted == torch.argmax(batch_y, dim=1)).sum().item()

    # Validation phase
    nn_model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for batch_x, batch_y in valid_loader:
            outputs = nn_model(batch_x)
            loss = criterion(outputs, torch.argmax(batch_y, dim=1))

            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += batch_y.size(0)
            val_correct += (predicted == torch.argmax(batch_y, dim=1)).sum().item()

    # Calculate metrics
    train_loss /= len(train_loader)
    val_loss /= len(valid_loader)
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total

    # Store history
    history['loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['accuracy'].append(train_acc)
    history['val_accuracy'].append(val_acc)

    # Early stopping check
    earlystop(val_acc)
    if earlystop.early_stop:
        break


history = HistoryWrapper(history)

plot_history(history)

# Prediction
nn_model.eval()
with torch.no_grad():
    pred_outputs = nn_model(testI)                    # verbose=False equivalent
    predY = torch.argmax(pred_outputs, dim=1).numpy()

acc = metrics.accuracy_score(testY, predY)
print("test accuracy:", acc)


## 2. Detection using CNN

Train an CNN to classify an image patch as face or not face.  Use  `vtrainI` and `vtrainY` as the training set and `validsetI` as the validation set.  You can try different architectures, and adjust values of the learning rates, number of iterations, early stopping, regularization, etc. to get a good result.  Use a large batch size (e.g., 50) to speed up the training time.  Remember to add the `callbacks` so that you can monitor the training process.

In [ ]:
### INSERT YOUR CODE HERE



_How does the MLP compare to the linear and non-linear classifiers that you tried in Tutorial 4?_
- **INSERT YOUR ANSWER HERE**

## 3. Data Augmentation

Now use data augmentation (introduced in the last tutorial) to try to improve the accuracy.

We can also add per-pixel noise or transformations. We define a few functions for adding per-pixel noise.  The following functions will add Gaussian pixel noise, add corruption noise (setting some input pixels to 0), scale and shift pixel values (changing contrast and brightness).

In [ ]:
def add_gauss_noise(X, sigma2=0.05):
    # add Gaussian noise with zero mean, and variance sigma2
    X = X.float()
    noise = torch.normal(0, sigma2, X.shape, dtype=torch.float32, device=X.device)
    return (X + noise).float()

def add_corrupt_noise(X, p=0.1):
    # apply pixel corruption (zero out value) with probability p
    X = X.float()
    mask = torch.rand(X.shape, dtype=torch.float32, device=X.device) > p
    return (X * mask.float()).float()

def add_scale_shift(X, sigma2=0.1, alpha2=0.2):
    # randomly scale and shift the pixel values (same for each image)
    # Xnew = a X + b
    # a is sampled from a Gaussian with mean 1, and variance sigma2
    # b is sampled from a Gaussian with mean 0, and variance alpha2
    X = X.float()

    if X.ndim == 3:
        dshape = (X.shape[0], 1, 1)
    elif X.ndim == 4:
        dshape = (X.shape[0], 1, 1, 1)
    else:
        dshape = (1,)

    a = torch.normal(1, sigma2, dshape, dtype=torch.float32, device=X.device)
    b = torch.normal(0, alpha2, dshape, dtype=torch.float32, device=X.device)

    result = torch.clamp(a * X + b, 0.0, 1.0)
    return result.float()

Next, we define a function for adding per-pixel noise (in this case just Gaussian noise). The noise is included using the `transforms.Compose`.

In [ ]:
# build the noise function
def addNoise(X):
    return add_gauss_noise(X, 0.04)

# build the data augmenter
transform = transforms.Compose([
    transforms.ToPILImage(),

    # Random rotation within ±10 degrees
    transforms.RandomRotation(10),

    # Random horizontal flipping
    transforms.RandomHorizontalFlip(),

    # Random affine transformation for width/height shift and shear
    transforms.RandomAffine(
        degrees=0,  # No additional rotation (already handled by RandomRotation)
        translate=(0.05, 0.05),  # Width and height shift (5% of image size)
        shear=5  # Shearing within ±5 degrees
    ),

    # Random zooming (simulated using RandomResizedCrop)
    transforms.RandomResizedCrop(size=(19, 19), scale=(0.95, 1.05)),  # Adjust size if needed

    # Convert PIL image to PyTorch tensor
    transforms.ToTensor(),

    # Add custom noise
    transforms.Lambda(addNoise)
])

Next we can show some examples of augmented images. Run the code below to see different random augmentations.

In [ ]:
img = trainI[4]
imgs = [img[0].detach().numpy()]

cnt = 0
while cnt < 5:
    # Apply augmentation - img already has correct shape (1, 19, 19)
    augmented = transform(img)

    # Convert to numpy for display (remove channel dimension)
    augmented_np = augmented[0].detach().numpy()

    imgs.append(augmented_np)
    cnt += 1

titles = ['original image', 'augmented', 'augmented', 'augmented', 'augmented', 'augmented']
plt.figure(figsize=(8,6))
show_imgs(imgs, nc=3, titles=titles)


The augmented images look similar to the original image, but contain small differences that the network can use to learn more about the class.

Now let's try training logistic regression with data augmentation.  We also disable early stopping so that the training sees more augmented data.

Customize dataset for augmentation and set some parameters

In [ ]:
class AugmentedTensorDataset(TensorDataset):
    def __init__(self, *tensors, transform=None):
        super().__init__(*tensors)
        self.transform = transform

    def __getitem__(self, index):
        img = self.tensors[0][index]
        label = self.tensors[1][index]

        if self.transform:
            img = self.transform(img)

        return img, label

batch_size = 50
epochs = 50
steps_per_epoch = len(vtrainI)/batch_size

# Create data loaders
train_dataset = AugmentedTensorDataset(trainI, trainYb, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_dataset = TensorDataset(validI, validYb)       # specify the validation set
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
torch.manual_seed(4487)
random.seed(4487)

# build the network for logistic regression
nn_model = nn.Sequential(
    nn.Flatten(),                                     # vectorize the input image
    nn.Linear(19*19*1, 2),                           # classification layer (2 classes)
    nn.Softmax(dim=1)
)


# compile and fit the network
criterion = nn.CrossEntropyLoss()                     # categorical_crossentropy equivalent
optimizer = optim.SGD(nn_model.parameters(), lr=0.05, momentum=0.9, nesterov=True)
                                                      # also calculate accuracy during training

history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}

epochs = 50

for epoch in range(epochs):
    # Training phase
    nn_model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch_x, batch_y in train_loader:
      optimizer.zero_grad()
      outputs = nn_model(batch_x)
      loss = criterion(outputs, torch.argmax(batch_y, dim=1))
      loss.backward()
      optimizer.step()

      train_loss += loss.item()
      _, predicted = torch.max(outputs.data, 1)
      train_total += batch_y.size(0)
      train_correct += (predicted == torch.argmax(batch_y, dim=1)).sum().item()

      # step_count += 1
      # if step_count >= steps_per_epoch:
      #   break

    # Validation phase
    nn_model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
      for batch_x, batch_y in valid_loader:
        outputs = nn_model(batch_x)
        loss = criterion(outputs, torch.argmax(batch_y, dim=1))

        val_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        val_total += batch_y.size(0)
        val_correct += (predicted == torch.argmax(batch_y, dim=1)).sum().item()

    # Calculate metrics
    avg_train_loss = train_loss/len(train_loader)
    avg_val_loss = val_loss / len(valid_loader)
    train_acc = train_correct / train_total
    val_acc = val_correct / val_total

    # Store history
    history['loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['accuracy'].append(train_acc)
    history['val_accuracy'].append(val_acc)


    print(f'Epoch {epoch+1}/{epochs} - '
      f'loss: {avg_train_loss:.4f} - '
      f'accuracy: {train_acc:.4f} - '
      f'val_loss: {avg_val_loss:.4f} - '
      f'val_accuracy: {val_acc:.4f}')


history = HistoryWrapper(history)



In [ ]:
plot_history(history)

# Prediction
nn_model.eval()
with torch.no_grad():
    pred_outputs = nn_model(testI)                    # verbose=False equivalent
    predY = torch.argmax(pred_outputs, dim=1).numpy()

acc = metrics.accuracy_score(testY, predY)
print("test accuracy:", acc)

Using data augmentation, the test accuracy improves from 0.60 to 0.70! (your numbers may be different)

Now train your best CNN from the previous section using data augmentation.
Try different per-pixel noise levels, and different options of the transforms,
and combinations of them.  Hopefully you should be able to improve the accuracy!

In [ ]:
### INSERT YOUR CODE HERE ###

_How does CNN with data augmentation compare with your work in Tutorial 8?_
- **INSERT YOUR ANSWER HERE**

# Test image
Now lets try your face detector on a real image.  Download the "nasa-small.png" image and put it in the same directory as your ipynb file.  The below code will load the image, crop out image patches and then extract features. (this may take a few minutes)

In [ ]:
fname = "nasa-small.png"

In [ ]:
# load image
testimg = skimage.io.imread(fname, as_gray=True)
print(testimg.shape)
plt.imshow(testimg, cmap='gray')

In [ ]:
# step size for the sliding window
step = 4

# extract window patches with step size of 4
patches = skimage.util.view_as_windows(testimg, (19,19), step=step)
psize = patches.shape
# collapse the first 2 dimensions
patches2 = patches.reshape((psize[0]*psize[1], psize[2], psize[3], 1))
print(patches2.shape)

# histogram equalize patches (improves contrast)
#newI = empty(patches2.shape)
#for i in range(patches2.shape[0]):
#    newI[i,:,:] = skimage.exposure.equalize_hist(patches2[i,:,:])
newI = patches2


Now predict using your classifier.  The extracted images are in `newI`.

In [ ]:
### YOUR CODE HERE

patches_tensor = torch.FloatTensor(patches2.transpose(0, 3, 1, 2))
with torch.no_grad():
    outputs = nn_model(patches_tensor)
    prednewY = torch.argmax(outputs, dim=1).numpy()

Now we we will view the results on the image.  Use the below code. `prednewY` is the vector of predictions.

In [ ]:
# reshape prediction to an image
imgY = prednewY.reshape(psize[0], psize[1])

# zoom back to image size
imgY2 = ndimage.interpolation.zoom(imgY, step, output=None, order=0)
# pad the top and left with half the window size
imgY2 = vstack((zeros((9, imgY2.shape[1])), imgY2))
imgY2 = hstack((zeros((imgY2.shape[0],9)), imgY2))
# pad right and bottom to same size as image
if (imgY2.shape[0] != testimg.shape[0]):
    imgY2 = vstack((imgY2, zeros((testimg.shape[0]-imgY2.shape[0], imgY2.shape[1]))))
if (imgY2.shape[1] != testimg.shape[1]):
    imgY2 = hstack((imgY2, zeros((imgY2.shape[0],testimg.shape[1]-imgY2.shape[1]))))

# show detections with image
#detimg = dstack(((0.5*imgY2+0.5)*testimg, 0.5*testimg, 0.5*testimg))
nimgY2 = 1-imgY2
tmp = nimgY2*testimg
detimg = dstack((imgY2+tmp, tmp, tmp))

# show it!
plt.figure(figsize=(9,9))
plt.subplot(2,1,1)
plt.imshow(imgY2, interpolation='nearest')
plt.title('detection map')
plt.subplot(2,1,2)
plt.imshow(detimg)
plt.title('image')
plt.axis('image')

_How did your face detector do compared to the last version?_
- **INSERT YOUR ANSWER HERE**

You can try it on your own images.  The faces should all be around 19x19 pixels though. We only used 1/8 of the training data. Try using more data to train it!